# DocMind pipeline walkthrough

This notebook runs each stage of the DocMind pipeline by hand on the sample PDFs so you can inspect the intermediate outputs. It uses the same functions as the app (`app/`), so nothing here is a re-implementation.

Requirements: the project dependencies plus Jupyter (`pip install jupyter`). Run `python evaluation/make_sample_docs.py` first if `evaluation/sample_docs/` is empty.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from app.ingestion import extract_pages
from app.preprocessing import preprocess_pages
from app.chunking import chunk_pages
from app.embeddings import Embedder

pdf_path = ROOT / "evaluation" / "sample_docs" / "remote_work_policy.pdf"
pdf_bytes = pdf_path.read_bytes()

## 1. Extraction (PyMuPDF): raw page text, before cleaning

In [ ]:
extraction = extract_pages(pdf_bytes, pdf_path.name)
print(extraction.page_count, "pages; empty:", extraction.empty_pages)
print(repr(extraction.pages[0].text[:400]))  # note the hard line breaks from the PDF layout

## 2. Preprocessing: wrapped lines joined, artifacts removed

In [ ]:
pages = preprocess_pages(extraction.pages)
print(pages[0].text[:400])

## 3. Chunking: try changing chunk_size / chunk_overlap and watch the chunks change

In [ ]:
chunks = chunk_pages(pages, chunk_size=800, chunk_overlap=150)
pd.DataFrame([{"chunk_id": c.chunk_id, "page": c.page_number, "chars": len(c.text), "start": c.text[:60]} for c in chunks])

## 4. Embeddings: each chunk becomes a unit-length 384-dimensional vector

In [ ]:
embedder = Embedder("sentence-transformers/all-MiniLM-L6-v2")
vectors = embedder.embed([c.text for c in chunks])
print(vectors.shape, "norms:", np.linalg.norm(vectors, axis=1).round(3))

## 5. Cosine similarity = dot product of unit vectors. This is exactly what FAISS `IndexFlatIP` computes.

In [ ]:
query = "Can I claim money for a desk?"
q = embedder.embed_query(query)[0]
scores = vectors @ q
for i in np.argsort(-scores):
    print(f"{scores[i]:.3f}  page {chunks[i].page_number}  {chunks[i].text[:70]}...")

## 6. Chunk-to-chunk similarity matrix: which chunks does the model consider related?

In [ ]:
pd.DataFrame(vectors @ vectors.T, index=[c.chunk_id.split(':',1)[1] for c in chunks],
             columns=[c.chunk_id.split(':',1)[1] for c in chunks]).round(2)